# Biography analysis
This code is made to get all the different analysis in the same place and enable more flexibility in analysis. It also get statistical analysis, running ALSI and conversions code directly using python.

## Imports and Relative Paths

In [1]:
import subprocess
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
BASE_DIR = Path().resolve()

GENERATION_DIR = BASE_DIR.parent / "generated_biographies"
RESOURCES_DIR  = BASE_DIR / "resources"
OUTPUT_DIR     = BASE_DIR / "outputs"

GENDER_DIR        = RESOURCES_DIR / "gender"
DISABILITIES_DIR  = RESOURCES_DIR / "disabilities"
NAME_ENTITIES_DIR = RESOURCES_DIR / "name_entities"
COMPLEXITY_DIR = RESOURCES_DIR / "complexity_tool" 
ALSI_DIR = COMPLEXITY_DIR / "ALSI-main"

## Complexity analysis

### Run ALSI on the texts and convert output to csv files for analysis

In [3]:
r_main = subprocess.run(
      ["Rscript", "R/main.R"],
      cwd=ALSI_DIR,
      capture_output=True, text=True, encoding="utf-8"
  )

In [4]:
r_conv = subprocess.run(
    ["Rscript", str(COMPLEXITY_DIR / "conversion.R")],
    cwd=ALSI_DIR / "out",
    capture_output=True, text=True, encoding="utf-8"
)

In [5]:
from resources.helper_functions import xlsx_to_csv
df_features = xlsx_to_csv(
    str(ALSI_DIR / "out" / "biographies_features_only.xlsx"),
    str(ALSI_DIR / "out" / "biographies_features_only_combined.csv")
)
print(df_features.shape)
print(df_features.columns.tolist())

(108, 124)
['doc_id', 'word_count', 'unique_word_count', 'content_word_count', 'unique_content_word_count', 'sentence_count', 'paragraph_count', 'avg_word_length', 'char_count', 'char_count_content', 'avg_sentence_length', 'avg_content_word_length', 'count_NA', 'count_ADJ', 'count_ADP', 'count_ADV', 'count_AUX', 'count_CCONJ', 'count_DET', 'count_INTJ', 'count_NOUN', 'count_NUM', 'count_PRON', 'count_PROPN', 'count_PUNCT', 'count_SCONJ', 'count_SYM', 'count_VERB', 'count_X', 'prop_NA', 'prop_ADJ', 'prop_ADP', 'prop_ADV', 'prop_AUX', 'prop_CCONJ', 'prop_DET', 'prop_INTJ', 'prop_NOUN', 'prop_NUM', 'prop_PRON', 'prop_PROPN', 'prop_PUNCT', 'prop_SCONJ', 'prop_SYM', 'prop_VERB', 'prop_X', 'present_count', 'past_count', 'future_count', 'conditional_count', 'subjunctive_count', 'indicative_count', 'imperative_count', 'infinitive_count', 'past_participle_count', 'present_participle_count', 'past_simple_count', 'present_prop', 'past_prop', 'future_prop', 'conditional_prop', 'subjunctive_prop', 

In [6]:
from resources.helper_functions import parse_doc_id
df_features = parse_doc_id(df_features)

### Statistical analysis of the corpus

In [7]:
from resources.statistics.statistics import compute_cohens_d_and_pvalue

In [ ]:
df_features_stats = compute_cohens_d_and_pvalue(df_features, "disability_in_prompt")
df_features_stats.to_csv(OUTPUT_DIR / "biographies_stats_complexity.csv")

## Gender Detection

In [9]:
from resources.gender.gender_detection import apply_gender_detection
df_gender = apply_gender_detection(GENERATION_DIR)
#df_gender.to_csv(OUTPUT_DIR / "gender_markers_bios.csv")
df_gender = parse_doc_id(df_gender)

## Disability Detection

In [10]:
from resources.disabilities.disabilities_detection import apply_disability_detection
df_dis = apply_disability_detection(GENERATION_DIR)
#df_dis.to_csv(OUTPUT_DIR / "disabilities_markers.csv")
df_dis = parse_doc_id(df_dis)

## NER Detection

In [11]:
from resources.name_entities.name_entities_detection import apply_ner_detection
df_ner = apply_ner_detection(GENERATION_DIR)
#df_ner.to_csv(OUTPUT_DIR / "ner_loc_orga_names.csv")
df_ner = parse_doc_id(df_ner)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
unique_df2 = df_dis.columns.difference(df_gender.columns)
unique_df3 = df_ner.columns.difference(df_gender.columns)

result = pd.concat([df_gender, df_dis[unique_df2], df_ner[unique_df3]], axis=1)
result.to_csv(OUTPUT_DIR / "gender_dis_ner_detection.csv")